# 현실의 데이터 11: 고객 문의 자동 분류

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/datashop/11_support_routing.ipynb)

**분야**: 플랫폼 · 서비스 &nbsp;|&nbsp; **이 분야에서 먼저 보는 문제**: 대화형 인공지능 · 고객응대 자동화 · 상담 보조 · 문의 분산처리

> 은행 고객 문의 1만 3천 개, 담당 77곳. 문의를 맞는 팀에 보낼 수 있을까?

**데이터**: Banking77 (약 1MB). PolyAI (Casanueva et al., 2020), 온라인 은행 고객 문의 13,083개를 의도 77가지로 나눈 영어 데이터 (GitHub PolyAI-LDN/task-specific-datasets).

인턴 · 산학협력 과제에서 처음 받게 될 데이터와 닮은 공개 데이터다. 데이터를 열어 보고, 아래 예시처럼 문제 하나를 정해 기준선과 비교한 뒤, 맨 끝의 **나의 다섯 칸** 을 채운다 (1주 2차시 '문제 정형화와 평가지표').

| 칸 | 이 노트북의 예시 |
|---|---|
| ① 한 행 | 고객 문의 한 문장 |
| ② 입력 X | 문장 |
| ③ 출력 y | 의도 77가지 중 하나 (보낼 팀) |
| ④ 평가지표 | 정확도 + 의도별 정확도 |
| ⑤ 기준선 | 가장 흔한 의도로 보내기 |

**논문 · 출처** (직접 열어 보자: 이 데이터를 누가, 왜, 어떻게 만들었나):
- [Casanueva et al. (2020) Efficient Intent Detection with Dual Sentence Encoders (Banking77)](https://arxiv.org/abs/2003.04807)

In [1]:
import io, os, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url, path):
    """Download url to path once (skip if it already exists)"""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (ML course data shop)"})
        with urllib.request.urlopen(req, timeout=600) as r, open(path, "wb") as f:
            f.write(r.read())
    return path

## 입장: 데이터 받기

In [2]:
BASE = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/"
train = pd.read_csv(download(BASE + "train.csv", "data/banking77_train.csv"))
test = pd.read_csv(download(BASE + "test.csv", "data/banking77_test.csv"))
print(train.shape, test.shape)
train.sample(5, random_state=0)

(10003, 2) (3080, 2)


,text,category
7145,How can I transfer money to a beneficiary?,beneficiary_not_allowed
834,How is my ATM deposit not available yet?,pending_cash_withdrawal
3585,There is a vendor name I don't recognize on a ...,card_payment_not_recognised
4497,My statement shows charges for things I never ...,compromised_card
7046,"Hi, I am interested buying crypto currency but...",beneficiary_not_allowed


## 열어 보기

의도가 몇 가지, 각각 몇 문장씩일까? 문장 길이는?

In [3]:
counts = train["category"].value_counts()
print("intents:", len(counts), "| most:", counts.index[0], counts.iloc[0], "| fewest:", counts.index[-1], counts.iloc[-1])
words = train["text"].str.split().str.len()
print("words per query: min", words.min(), "median", int(words.median()), "max", words.max())
for cat in ["card_arrival", "lost_or_stolen_card", "exchange_rate"]:
    print(f"\n[{cat}]", train.loc[train["category"] == cat, "text"].head(3).tolist())

intents: 77 | most: card_payment_fee_charged 187 | fewest: contactless_not_working 35
words per query: min 2 median 10 max 79

[card_arrival] ['I am still waiting on my card?', "What can I do if my card still hasn't arrived after 2 weeks?", 'I have been waiting over a week. Is the card still coming?']

[lost_or_stolen_card] ['Has there been any activity on my card today?', 'I lost my wallet and all my cards were in it.', "I'm panicking!  I lost my card!  Help!"]

[exchange_rate] ['What is my money worth in other countries?', 'Will my money be of equal value when I travel abroad?', 'Do you know the rate of exchange?']


## 문제 하나 골라 보기: 문의 → 보낼 팀 (77가지 분류)

문장을 단어 빈도 (TF-IDF) 벡터로 바꾸고 로지스틱회귀로 분류한다. 기준선 = 늘 가장 흔한 의도.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

most_common = train["category"].value_counts().idxmax()
model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=2000)).fit(train["text"], train["category"])
pred = model.predict(test["text"])
print(f"baseline (always '{most_common}'): accuracy {np.mean(test['category'] == most_common):.1%}")
print(f"TF-IDF + logistic regression : accuracy {np.mean(test['category'] == pred):.1%}")

per_intent = pd.Series(test["category"].values == pred).groupby(test["category"].values).mean().sort_values()
print("\nhardest intents:\n", per_intent.head(5).round(2))

baseline (always 'card_payment_fee_charged'): accuracy 1.3%
TF-IDF + logistic regression : accuracy 85.5%

hardest intents:
 virtual_card_not_working                   0.30
pending_transfer                           0.60
card_acceptance                            0.65
wrong_exchange_rate_for_cash_withdrawal    0.68
contactless_not_working                    0.68
dtype: float64


In [5]:
wrong = pd.DataFrame({"true": test["category"].values, "pred": pred})
wrong = wrong[wrong["true"] != wrong["pred"]]
pairs = wrong.value_counts().head(5)
print("most confused (true -> predicted):")
for (t, p), n in pairs.items():
    print(f"  {t:35s} -> {p:35s} {n}")

most confused (true -> predicted):
  virtual_card_not_working            -> getting_virtual_card                13
  virtual_card_not_working            -> get_disposable_virtual_card         9
  card_acceptance                     -> card_not_working                    8
  why_verify_identity                 -> verify_my_identity                  6
  unable_to_verify_identity           -> verify_my_identity                  6


## 다른 문제로 바꿔 보기

- **문의 분산처리**: 77개 의도를 실제 팀 5~6개로 묶으면 문제가 쉬워질까? 묶는 방법은 누가 정하나?
- **상담 보조**: '맞는 팀' 대신 '상담원에게 보여 줄 비슷한 과거 문의 3개' 를 출력으로 정하면 지표는?
- **고객응대 자동화**: 자신 없는 문의 (예측 확률이 낮은 것) 만 사람에게 넘기면, 자동 처리율과 정확도는 어떻게 바뀔까?

### 나의 다섯 칸

아래 표를 직접 채워 보자. 위의 예시와 다르게 정해도 된다.

| 칸 | 내가 정한 것 |
|---|---|
| ① 한 행은 무엇? | |
| ② 입력 X (예측할 때 알 수 있는 것만) | |
| ③ 출력 y | |
| ④ 평가지표 | |
| ⑤ 기준선 | |